<a href="https://colab.research.google.com/github/lldb14/Sinais-Biol-gicas/blob/main/Tarefa_1_Sinais_Biol%C3%B3gicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Um conjunto de dados de frequência cardíaca instantânea e acelerometria de várias noites com rótulos de estágio do sono por EEG.

Dados de sono de 47 voluntários adultos saudáveis, sem histórico de distúrbios do sono, recrutados na comunidade local por meio de panfletos de divulgação do estudo. O protocolo do estudo foi aprovado pelo Comitê de Ética em Pesquisa da Universidade de Massachusetts Lowell. Os participantes foram selecionados sem levar em consideração sexo ou etnia e forneceram consentimento livre e esclarecido por escrito. Cada participante usou simultaneamente uma faixa de cabeça Dreem 2 e um Apple Watch por até sete noites consecutivas, totalizando 253 noites de dados.

O conjunto de dados contém gravações de várias noites da frequência cardíaca instantânea e da acelerometria de 3 eixos do Apple Watch, juntamente com rótulos dos estágios do sono derivados do EEG Dreem 2 e classificados de acordo com o padrão AASM (Vigília, N1, N2, N3, REM).

Este recurso apoia o desenvolvimento e a validação de métodos de classificação dos estágios do sono baseados em dispositivos vestíveis, a modelagem multimodal de sinais de frequência cardíaca e movimento e pesquisas que se beneficiam de dados de sono de várias noites.

Song, T. (2026). A Multi-Night Instantaneous Heart Rate and Accelerometry Dataset with EEG Sleep Stage Labels (version 1.0.0). PhysioNet. RRID:SCR_007345. https://doi.org/10.13026/a0sy-7t69

## Dados

O conjunto de dados está organizado em 47 pastas de nível de sujeito, com cada pasta contendo entre 3 e 7 noites de dados brutos. Para cada noite, são fornecidos três arquivos: *motion.csv, hr.csv*, e *labels.mat*.

1. motion.csv : Dados do acelerômetro de três eixos (x, y, z) registrados pelo Apple Watch. Cada linha inclui um registro de data e hora em tempo Unix (segundos, com precisão de subsegundos) indicando quando a medição foi registrada.

2. hr.csv : Valores de frequência cardíaca instantânea (FCI) derivados do sensor PPG do Apple Watch via HealthKit. A FCI é medida em batimentos por minuto (bpm) em pontos de tempo específicos, com uma taxa de amostragem aproximada de 0,2 Hz. Cada linha inclui um registro de data e hora em tempo Unix (segundos, com precisão de subsegundos).

3. labels.mat : Contém três variáveis-chave:
- recStart: Marca de tempo que indica o início da gravação.

- dreem_labelAnotações automatizadas dos estágios do sono obtidas pelo dispositivo Dreem, codificadas como números inteiros (0 = Acordado, 1 = N1, 2 = N2, 3 = N3, 4 = REM, 5 = Desconhecido).

- expert_label Anotações manuais das fases do sono feitas por um especialista em sono usando o mesmo esquema de codificação.

https://physionet.org/content/bidsleep-dataset/1.0.0/

# Amostra

- HR: como a amostragem é de ~0,2 Hz (1 amostra a cada ~5 s), uma janela curta (ex: 30 s) mostraria só 5–6 pontos — pouco informativo. Por isso usei 10 minutos (600 s), o suficiente para ver oscilações da frequência cardíaca ao longo de um trecho do sono.

- Motion: o acelerômetro tem taxa de amostragem bem mais alta, então a mesma janela de 600 s também funciona bem para ver eventos de movimento (picos = agitação/mudança de posição) e trechos de repouso (linha quase reta).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# 1. Baixar os dados de um sujeito (ex: Bidslab00)
!wget -r -N -c -np -q --show-progress \
  https://physionet.org/files/bidsleep-dataset/1.0.0/Bidslab00/

# 2. Conferir se baixou
!find physionet.org/files/bidsleep-dataset/1.0.0/Bidslab00 -type f

# --- Carregar os dados ---
hr = pd.read_csv("physionet.org/files/bidsleep-dataset/1.0.0/Bidslab00/1/hr.csv")
motion = pd.read_csv("physionet.org/files/bidsleep-dataset/1.0.0/Bidslab00/1/motion.csv")

# Renomeie as colunas conforme o que aparecer no seu CSV real
# Supondo que a 1ª coluna seja tempo (Unix) e a 2ª seja o valor
hr.columns = ["time", "bpm"]
motion.columns = ["time", "x", "y", "z"]

# --- Converter tempo Unix para segundos relativos ao início do sinal ---
hr["t_rel"] = hr["time"] - hr["time"].iloc[0]
motion["t_rel"] = motion["time"] - motion["time"].iloc[0]

# --- Escolher uma janela de tempo que mostre bem as características ---
# HR tem taxa de amostragem baixa (~0.2 Hz), então uma janela maior (ex: 10 min = 600 s)
# ajuda a ver a variação natural do sinal
t_ini, t_fim = 0, 600  # segundos

hr_win = hr[(hr["t_rel"] >= t_ini) & (hr["t_rel"] <= t_fim)]
motion_win = motion[(motion["t_rel"] >= t_ini) & (motion["t_rel"] <= t_fim)]


In [ ]:
# --- Plot: Frequência cardíaca ---
plt.figure(figsize=(12, 4))
plt.plot(hr_win["t_rel"], hr_win["bpm"], color="crimson", marker="o", markersize=3, linewidth=1)
plt.xlabel("Tempo (s)")
plt.ylabel("Frequência cardíaca (bpm)")
plt.title(f"Frequência cardíaca instantânea — janela de {t_fim - t_ini}s")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot: Acelerometria (3 eixos) ---
plt.figure(figsize=(12, 4))
plt.plot(motion_win["t_rel"], motion_win["x"], label="Eixo X", linewidth=0.8)
plt.plot(motion_win["t_rel"], motion_win["y"], label="Eixo Y", linewidth=0.8)
plt.plot(motion_win["t_rel"], motion_win["z"], label="Eixo Z", linewidth=0.8)
plt.xlabel("Tempo (s)")
plt.ylabel("Aceleração (g)")  # confirme a unidade no seu dado; Apple Watch geralmente reporta em g
plt.title(f"Acelerometria 3 eixos — janela de {t_fim - t_ini}s")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()